# VLS ↔ GMD bunch alignment

Which Gotthard line belongs to which FEL pulse? `write_h5.py` stores Gotthard line `b + VLS_OFFSET` as bunch `b` and keeps
`TRAIN_LENGTH` bunch slots (`config.VLS_OFFSET`, `config.TRAIN_LENGTH`). This notebook checks both on the raw files of one
run.

**Steps.**
1. Read the GMD and the full Gotthard frames (every line, zero offset) from the raw files. Integrate each line over
   `VLS_ROI` after subtracting its median over `VLS_BG_PIXELS` (common mode).
2. Offset scan: correlation between the line integral and the GMD of bunch `b` for line `b + offset`, after removing the
   mean train and bunch-position trends.
3. Lit lines: mean integral of each line next to the GMD train profile, and each line's correlation with its GMD bunch.
4. Which lines the combined file keeps, and (optionally) the same checks on a combined file: the stored VLS, the TOF hits
   and the GMD on the bunch axis.

In [ ]:
# ---- Parameters ---------------------------------------------------------------
RUN = 60559                  # raw FLASH run
TRAIN_STEP = 20              # every N-th train (a raw frame is ~0.5 MB as float32)
MAX_FILES = None             # first N raw files only; None = all
OFFSET = None                # offset to check; None = config.VLS_OFFSET
OFFSETS = range(-10, 16)     # offsets scanned
VLS_ROI = (200, 650)         # pixels integrated (half-open); 200-650 holds the 265-282 eV peak of run 60559
VLS_BG_PIXELS = (650, 800)   # signal-free pixels for the per-line common mode
VLS_BAD_PIXELS = [0, 1, 291, 292, 679, 680, 681, *range(832, 864)]   # 291-292: hot, found on run 60559
LIT_FRACTION = 0.03          # a line counts as lit when its mean integral exceeds this fraction of the largest

# Optional combined file from write_h5.py (relative to config.COMBINED_DIR or absolute); None = skip section 4b
COMBINED_H5 = "run60559_Ethanol_undulator_scan_2807989445.h5"
E_TOF_MAX = 1000             # ns; electron hits counted per slot below this (later ones include ion crosstalk)

In [ ]:
# ---- Imports and paths ------------------------------------------------------
from pathlib import Path
import glob
import sys
import warnings

import h5py
import matplotlib.pyplot as plt
import numpy as np

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "analysis" / "scripts" / "config.py").exists())
sys.path.insert(0, str(REPO_ROOT / "analysis" / "scripts"))
import config
import data_loading

for msg in ("All-NaN slice", "Mean of empty slice", "invalid value encountered", "Degrees of freedom"):
    warnings.filterwarnings("ignore", message=msg)
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3})
OFFSET = config.VLS_OFFSET if OFFSET is None else OFFSET
print(f"checking offset {OFFSET} (config.VLS_OFFSET = {config.VLS_OFFSET}), TRAIN_LENGTH = {config.TRAIN_LENGTH}")

## 1. Load the raw GMD and Gotthard frames

`load_raw_h5` with `vls_offset=0` and `train_length` = the number of Gotthard lines keeps every line; GMD slots beyond the
pulses are NaN. `I` holds the line integrals, `(n_trains, n_lines)`; `G` the GMD, `(n_trains, n_bunches)`.

In [ ]:
raw_files = sorted(glob.glob(str(config.RAW_H5_DIR / f"*run{RUN}_*.h5")))
with h5py.File(raw_files[0], "r") as f:
    n_lines = f["/FL2/Support Infrastructure/Gotthard/images/value"].shape[1]
    n_bunches = f["/FL2/Photon Diagnostic/GMD/Pulse resolved energy/energy hall/value"].shape[2]
raw = data_loading.load_raw_h5(RUN, config=1, max_files=MAX_FILES, train_step=TRAIN_STEP,
                               train_length=n_lines, vls_offset=0)
v = raw.vls                                           # (n_trains, n_lines, n_pixels), NaN = no frame
v[..., VLS_BAD_PIXELS] = np.nan
cm = np.nanmedian(v[..., slice(*VLS_BG_PIXELS)], axis=-1, keepdims=True)
I = np.nansum(v[..., slice(*VLS_ROI)] - cm, axis=-1)  # line integrals (ADU)
I[~np.isfinite(v[..., VLS_ROI[0]])] = np.nan
G = raw.gmd[:, :n_bunches]
del v, cm
has_pulse = np.isfinite(G).mean(axis=0) > 0.5
print(f"run {RUN}: {len(G)} trains, {n_lines} Gotthard lines per frame, {n_bunches} GMD slots "
      f"({has_pulse.sum()} with pulses: {np.flatnonzero(has_pulse)[[0, -1]]})")

## 2. Offset scan

For each offset, pair line `b + offset` with GMD bunch `b` and correlate over all shots, after subtracting from both the
mean of each train and the mean of each bunch position. The right offset stands out; at any other the remaining
correlation is small.

In [ ]:
def demean(a):
    # remove the mean of each train and of each bunch position
    return a - np.nanmean(a, axis=0) - np.nanmean(a, axis=1, keepdims=True)


def offset_r(G, I, offset):
    b = np.arange(G.shape[1])
    ok = (b + offset >= 0) & (b + offset < I.shape[1])
    g, x = G[:, b[ok]], I[:, b[ok] + offset]
    valid = np.isfinite(g) & np.isfinite(x) & (g > 0)
    g, x = demean(np.where(valid, g, np.nan)), demean(np.where(valid, x, np.nan))
    m = np.isfinite(g) & np.isfinite(x)
    return np.corrcoef(g[m], x[m])[0, 1] if m.sum() > 10 else np.nan


offsets = np.array(list(OFFSETS))
r_off = np.array([offset_r(G, I, o) for o in offsets])
best = int(offsets[np.nanargmax(r_off)])
others = r_off[offsets != best]
print(f"best offset {best}: r = {np.nanmax(r_off):.3f}; largest r at any other offset {np.nanmax(others):.3f}")
print(f"offset {OFFSET} (checked): r = {r_off[offsets == OFFSET][0]:.3f}")

fig, ax = plt.subplots(figsize=(8, 3.8))
ax.plot(offsets, r_off, "o-")
ax.axvline(OFFSET, color="C3", ls="--", label=f"checked offset {OFFSET}")
ax.set(xlabel="offset (Gotthard line − GMD bunch)", ylabel="Pearson r", title=f"Run {RUN}: Gotthard / GMD offset scan")
ax.legend()
fig.tight_layout()
plt.show()

## 3. Which lines carry a pulse

Top: mean integral of each Gotthard line (minus the median of the lines that have no GMD bunch at this offset) and the
mean GMD of bunch `b`, drawn at line `b + OFFSET` and scaled to the same height. A line counts as lit when its mean
integral exceeds `LIT_FRACTION` of the largest. Bottom: correlation of each line with the GMD of bunch `line − OFFSET`
(after removing the train mean of the GMD); lines without a bunch are compared with the nearest one (open markers). The
correlation fades towards the end of the train, where the pulses are weak and the GMD's own noise dominates their
fluctuations, so it does not decide which lines are lit. Shaded: the lines the combined file keeps (`OFFSET` …
`OFFSET + TRAIN_LENGTH − 1`).

In [ ]:
lines = np.arange(n_lines)
partner = lines - OFFSET
has_partner = (partner >= 0) & (partner < n_bunches)
g_res = G - np.nanmean(G, axis=1, keepdims=True)          # bunch-to-bunch fluctuation within each train
r_line = np.full(n_lines, np.nan)
for l in lines:
    b = int(np.clip(partner[l], 0, n_bunches - 1))
    m = np.isfinite(I[:, l]) & np.isfinite(g_res[:, b])
    r_line[l] = np.corrcoef(I[m, l], g_res[m, b])[0, 1]
prof = np.nanmean(I, axis=0)
prof -= np.median(prof[~has_partner]) if (~has_partner).any() else 0
g_prof = np.nanmean(G, axis=0)
lit = prof > LIT_FRACTION * np.nanmax(prof)
stored = (lines >= OFFSET) & (lines < OFFSET + config.TRAIN_LENGTH)

fig, axs = plt.subplots(2, 1, figsize=(13, 6.5), sharex=True)
axs[0].bar(lines, prof, width=0.8, color=np.where(lit, "C0", "0.6"), label="Gotthard line integral (grey: not lit)")
axs[0].plot(np.arange(n_bunches) + OFFSET, g_prof / np.nanmax(g_prof) * np.nanmax(prof), "C3.-", ms=3,
            label=f"GMD of bunch (line − {OFFSET}), scaled")
axs[0].set(ylabel="mean integral (ADU)")
axs[0].legend(loc="upper right")
axs[1].plot(lines[has_partner], r_line[has_partner], "o", ms=4, label="paired with its GMD bunch")
axs[1].plot(lines[~has_partner], r_line[~has_partner], "o", mfc="none", ms=6, label="no bunch: nearest one")
axs[0].axhline(LIT_FRACTION * np.nanmax(prof), color="k", ls=":", lw=1)
axs[1].set(xlabel="Gotthard line", ylabel="r with GMD")
axs[1].legend(loc="upper right")
for ax in axs:
    ax.axvspan(OFFSET - 0.5, OFFSET + config.TRAIN_LENGTH - 0.5, color="C2", alpha=0.08)
fig.suptitle(f"Run {RUN}, offset {OFFSET}: shaded = lines stored as bunch slots 0–{config.TRAIN_LENGTH - 1}")
fig.tight_layout()
plt.show()

print(f"lit lines: {lines[lit].min()}-{lines[lit].max()} ({lit.sum()} lines; GMD pulses: {has_pulse.sum()})")
print(f"not lit: {lines[~lit].tolist()}")
print("mean integral relative to the largest, lines " + ", ".join(
    f"{l}: {prof[l] / np.nanmax(prof):.3f}" for l in [*range(4), *range(n_lines - 6, n_lines)]))

## 4. What the combined file keeps

With `OFFSET` and `TRAIN_LENGTH`, line `OFFSET + s` goes to bunch slot `s`. Every lit line should be stored, each in the
slot of its own GMD bunch.

In [ ]:
lit_not_stored = lines[lit & ~stored]
dark_stored = lines[~lit & stored]
print(f"stored lines {OFFSET}-{OFFSET + config.TRAIN_LENGTH - 1} -> slots 0-{config.TRAIN_LENGTH - 1}")
print(f"lit lines not stored: {lit_not_stored.tolist() or 'none'}")
print(f"stored lines without a pulse: {dark_stored.tolist()} -> slots {(dark_stored - OFFSET).tolist()}")
print(f"lines dropped: {lines[~stored].tolist()}")

### 4b. Check on a combined file

The same offset scan on the stored VLS, now as an extra shift between the stored slot and the GMD slot (0 means the
file is aligned), and the hits per shot of both TOFs, the fraction of shots with a GMD value and the stored VLS line
integral in every bunch slot.

In [ ]:
if COMBINED_H5 is not None:
    cpath = Path(COMBINED_H5)
    cpath = cpath if cpath.is_absolute() else config.COMBINED_DIR / cpath
    with h5py.File(cpath, "r") as f:
        rows = np.arange(0, f["tID"].shape[0], TRAIN_STEP)
        gc = f["gmd"][rows]
        print(f"{cpath.name}: vls_offset attribute = {f['vls'].attrs.get('vls_offset')}, {f['gmd'].shape[1]} slots")
        vc = f["vls"][rows].astype(np.float32)
        vc[..., VLS_BAD_PIXELS] = np.nan
        Ic = np.nansum(vc[..., slice(*VLS_ROI)] - np.nanmedian(vc[..., slice(*VLS_BG_PIXELS)], -1, keepdims=True), -1)
        Ic[~np.isfinite(vc[..., VLS_ROI[0]])] = np.nan
        del vc
        te, ti = f["tofs_e"][rows], f["tofs_i"][rows]
    n_e = ((te > 0) & (te < E_TOF_MAX * 10)).sum(-1)           # TDC units of 100 ps
    n_i = ((ti > 0) & (ti < 40000)).sum(-1)
    shifts = np.arange(-3, 4)
    r_shift = [offset_r(gc, Ic, s) for s in shifts]
    print("stored VLS vs GMD, extra shift: " + "  ".join(f"{s:+d}: {r:.2f}" for s, r in zip(shifts, r_shift)))

    slots = np.arange(gc.shape[1])
    ic_prof = np.nanmean(Ic, axis=0)
    ic_prof -= np.nanmin(ic_prof)
    fig, ax = plt.subplots(figsize=(13, 4))
    for y, lab in [(np.isfinite(gc).mean(0), "shots with GMD"), (n_e.mean(0), f"e hits / shot (< {E_TOF_MAX} ns)"),
                   (n_i.mean(0), "ion hits / shot"), (ic_prof, "stored VLS line integral")]:
        ax.plot(slots, y / np.nanmax(y), ".-", ms=4, label=lab)
    ax.set(xlabel="bunch slot", ylabel="normalised to max", title=f"{cpath.name}: bunch axis of every stream")
    ax.legend()
    fig.tight_layout()
    plt.show()
    last = gc.shape[1] - 1
    print(f"slot {last}: GMD in {np.isfinite(gc[:, last]).mean():.3f} of shots; e hits/shot {n_e[:, last].mean():.3f} "
          f"vs slots 0-{last - 1} {n_e[:, :last].mean():.3f}; ion {n_i[:, last].mean():.3f} vs {n_i[:, :last].mean():.3f}")